# 02 – Feature Table and Split

In this notebook we build the table that all our M2 experiments and models will use.

Each row is one student in one subject (Math, English, or Science).
What we want to predict is whether that student failed the subject in term 3.

We only use information from before term 3, so the model can't "see" the answer
ahead of time.

## Step 1 – Load the cleaned data

In [1]:
# Import libraries
import pandas as pd
import numpy as np

In [2]:
# Term 3 starts on this date (from the data dictionary)
TERM3_START = "2026-02-18"

# The three subjects that have tutoring
SUBJECTS = ["Mathematics", "English Language Arts", "Science"]

In [3]:
# Folder paths 
CLEAN = "../data/clean/"
RAW = "../data/raw/"

In [4]:
# Load the cleaned files from M1 (student_id is read as text to keep leading zeros)

students  = pd.read_csv(CLEAN + "students_clean.csv", dtype={"student_id": str})
grades    = pd.read_csv(CLEAN + "term_grades_clean.csv.gz", dtype={"student_id": str})
referrals = pd.read_csv(CLEAN + "tutoring_referrals_clean.csv", dtype={"student_id": str})
sessions  = pd.read_csv(CLEAN + "tutoring_sessions_clean.csv.gz", dtype={"student_id": str})

In [5]:
# Courses file is only in raw, we need it to know each course's subject
courses = pd.read_csv(RAW + "courses.csv")

In [6]:
# Check the row counts
print("students: ", len(students))
print("grades:   ", len(grades))
print("referrals:", len(referrals))
print("sessions: ", len(sessions))
print("courses:  ", len(courses))

students:  40000
grades:    720000
referrals: 14917
sessions:  413370
courses:   32


In [7]:
# Check the columns we made in M1 are there
print("\nGrades has 'failed':", "failed" in grades.columns)
print("Sessions has 'term':", "term" in sessions.columns)


Grades has 'failed': True
Sessions has 'term': True


**Result:** All files loaded with the same counts as the end of the cleaning
notebook (students 40,000; grades 720,000; referrals 14,917; sessions 413,370).
The `failed` column and the session `term` column from M1 are both there.

## Step 2 – Link grades to subjects
The grades file only shows a course code (like MTH110), not the subject.
So we add the subject from the courses file. We only keep Math, English and
Science, because those are the subjects with tutoring.

We also check if any student has more than one course in the same subject.
If they do, we will need to combine those courses later.

In [8]:
# Add the subject of each course to the grades
grades = grades.merge(courses[["course_code", "subject_area"]], on="course_code", how="left")
print("Grades with no subject found:", grades["subject_area"].isna().sum())

Grades with no subject found: 0


In [9]:
# Keep only the three tutoring subjects
core = grades[grades["subject_area"].isin(SUBJECTS)].copy()
print("Grade rows in the 3 tutoring subjects:", len(core))

Grade rows in the 3 tutoring subjects: 360000


In [10]:
# Count how many different courses each student has in each subject
courses_per_subject = core.groupby(["student_id", "subject_area"])["course_code"].nunique()
print("\nCourses per student per subject:")
print(courses_per_subject.value_counts())


Courses per student per subject:
course_code
1    120000
Name: count, dtype: int64


In [11]:

# Count how many students take each subject
print("\nStudents per subject:")
print(core.groupby("subject_area")["student_id"].nunique())


Students per subject:
subject_area
English Language Arts    40000
Mathematics              40000
Science                  40000
Name: student_id, dtype: int64


**Result:** 360,000 grade rows are in Math, English and Science
(40,000 students × 3 subjects × 3 terms). Every course matched a subject.
Each student has exactly 1 course in each subject, so we don't need to
combine any courses. All 40,000 students take all three subjects.

## Step 3 – Make one row per student and subject
Right now the grades table has one row per student, course and term.
We want one row per student and subject, with their result in each term.

If a student has more than one course in a subject, we count them as
failing that term if they failed any of those courses.

In [12]:
# For each student, subject and term: did they fail any course in that subject?
per_term = (core.groupby(["student_id", "subject_area", "term"])["failed"]
                .max()          # True if they failed at least one course
                .reset_index())

In [13]:
# Turn the terms into columns: failed_t1, failed_t2, failed_t3
base = per_term.pivot(index=["student_id", "subject_area"], columns="term", values="failed")
base.columns = [f"failed_t{t}" for t in base.columns]
base = base.reset_index()

In [14]:
# Check the size of the new table
print("Rows (student x subject):", len(base))
print("Different students:", base["student_id"].nunique())
print("\nRows per subject:")
print(base["subject_area"].value_counts())

Rows (student x subject): 120000
Different students: 40000

Rows per subject:
subject_area
English Language Arts    40000
Mathematics              40000
Science                  40000
Name: count, dtype: int64


In [15]:
# Check for missing term results
print("\nMissing values:")
print(base[["failed_t1", "failed_t2", "failed_t3"]].isna().sum())

base.head()


Missing values:
failed_t1    0
failed_t2    0
failed_t3    0
dtype: int64


,student_id,subject_area,failed_t1,failed_t2,failed_t3
0,2100012,English Language Arts,False,False,False
1,2100012,Mathematics,False,False,False
2,2100012,Science,False,False,False
3,2100013,English Language Arts,False,False,False
4,2100013,Mathematics,False,False,False


**Result:** The table has 120,000 rows (40,000 students × 3 subjects).
No term results are missing.

We want to predict `failed_t3` (did the student fail in term 3).
We can use `failed_t1` to help, because term 1 happens before term 3.

## Step 4 – Add student information
Now we add information about each student from the students file:
their prior-year GPA, whether they ride the bus, and some background
information. All of this is known before term 3.

We also split students into four GPA groups (quartiles), because our
hypotheses compare students with similar past grades. Students with no
prior GPA get their own group, "Missing".

In [16]:
# Columns we want from the students file
student_cols = ["student_id", "school_code", "grade_level", "prior_year_gpa", "gpa_missing",
                "bus_rider", "econ_disadvantaged", "english_learner", "iep"]

In [17]:
# Add student information to each row
features = base.merge(students[student_cols], on="student_id", how="left")
print("Rows after adding student info:", len(features))

Rows after adding student info: 120000


In [18]:
# Change Y/N columns into 1/0 so models can use them
for col in ["bus_rider", "econ_disadvantaged", "english_learner", "iep"]:
    features[col] = (features[col] == "Y").astype(int)

In [19]:
# Make 4 GPA groups (quartiles) from students who have a GPA
features["gpa_group"] = pd.qcut(features["prior_year_gpa"], q=4,
                                labels=["Q1 (lowest)", "Q2", "Q3", "Q4 (highest)"])

In [20]:
# Students with no GPA get their own group
features["gpa_group"] = features["gpa_group"].cat.add_categories("Missing")
features.loc[features["gpa_missing"], "gpa_group"] = "Missing"

In [21]:
# Check how many rows are in each GPA group
print("\nRows per GPA group:")
print(features["gpa_group"].value_counts().sort_index())


Rows per GPA group:
gpa_group
Q1 (lowest)     29562
Q2              28746
Q3              29070
Q4 (highest)    29013
Missing          3609
Name: count, dtype: int64


In [22]:
# Show the GPA range of each group
print("\nGPA range per group:")
print(features.groupby("gpa_group", observed=True)["prior_year_gpa"].agg(["min", "max"]))


GPA range per group:
               min   max
gpa_group               
Q1 (lowest)   0.00  2.37
Q2            2.38  2.83
Q3            2.84  3.30
Q4 (highest)  3.31  4.00
Missing        NaN   NaN


**Result:** Still 120,000 rows, so no rows were lost. The four GPA groups have
about 29,000 rows each. 3,609 rows are in the "Missing" group
(1,203 students who joined in 2025 × 3 subjects).

GPA groups: Q1 = 0.00 to 2.37, Q2 = 2.38 to 2.83, Q3 = 2.84 to 3.30,
Q4 = 3.31 to 4.00.

## Step 5 – Was the student referred before term 3?
We add whether each student was referred to tutoring in that subject.

We only count a referral if it happened before term 3 started (Feb 18, 2026).
If a student was referred during term 3, it may be because they were already
failing, so it would mix up cause and effect.

In [23]:
# Keep the columns we need from referrals
ref = referrals[["student_id", "subject_area", "referral_date", "referred_by"]].copy()

In [24]:
# Remove old referral columns first, so running this cell again doesn't create duplicates
features = features.drop(columns=["referral_date", "referred_by", "referred_any",
                                  "referred_before_t3", "referred_during_t3"], errors="ignore")

# Add referral info to each student x subject row (no referral = empty)
features = features.merge(ref, on=["student_id", "subject_area"], how="left")
print("Rows after adding referrals:", len(features))

Rows after adding referrals: 120000


In [25]:
# Referred at any time in the year
features["referred_any"] = features["referral_date"].notna()

In [26]:
# Referred before term 3 started (this is the one we use)
features["referred_before_t3"] = features["referral_date"] < TERM3_START

In [27]:
# Mark students referred during term 3, so experiments can leave them out or report them separately
features["referred_during_t3"] = features["referred_any"] & ~features["referred_before_t3"]

In [28]:
# Check the counts
print("Referred at any time:", features["referred_any"].sum())
print("Referred before term 3:", features["referred_before_t3"].sum())
print("Referred during term 3:", features["referred_during_t3"].sum())

Referred at any time: 14917
Referred before term 3: 13538
Referred during term 3: 1379


In [29]:
# Who made the referrals (before term 3)?
print("Who referred (before term 3):")
print(features.loc[features["referred_before_t3"], "referred_by"].value_counts())

Who referred (before term 3):
referred_by
Teacher      9375
Self         1912
Parent       1239
Counselor    1012
Name: count, dtype: int64


**Result:** Still 120,000 rows. 14,917 student-subject rows were referred at
some point in the year: 13,538 before term 3 and 1,379 during term 3.
We only count referrals before term 3.

Most referrals before term 3 came from teachers (9,375), then self (1,912),
parents (1,239) and counselors (1,012).

Students referred during term 3 (1,379) are marked with `referred_during_t3`.
They may have been referred because they were already failing, so the
experiments should leave them out of the "not referred" group or report
them separately.

## Step 6 – Tutoring attendance before term 3
For H2 we need to know how much tutoring each student actually got before
term 3. We only count sessions before term 3 started, so we don't use
anything from the same time as the outcome.

For each student and subject we count:
- how many sessions were scheduled
- how many they attended
- the attendance rate (attended ÷ scheduled)

Students with no sessions scheduled get no attendance rate, because you
can't attend sessions that never happened.

In [30]:
# Remove attendance columns if this cell was already run (so it can be run again)
features = features.drop(columns=["sessions_scheduled_before_t3", "sessions_attended_before_t3",
                                  "attendance_rate_before_t3", "attendance_group"], errors="ignore")

# Keep only sessions before term 3 started
before_t3 = sessions[sessions["session_date"] < TERM3_START].copy()
print("Sessions before term 3:", len(before_t3), "of", len(sessions))

# Mark each session as attended (1) or not (0)
before_t3["attended"] = (before_t3["attendance"] == "Present").astype(int)

# For each student and subject: count scheduled and attended sessions
att = (before_t3.groupby(["student_id", "subject_area"])
                .agg(sessions_scheduled_before_t3=("attended", "size"),
                     sessions_attended_before_t3=("attended", "sum"))
                .reset_index())

# Add these counts to the feature table (no sessions = 0)
features = features.merge(att, on=["student_id", "subject_area"], how="left")
features[["sessions_scheduled_before_t3", "sessions_attended_before_t3"]] = (
    features[["sessions_scheduled_before_t3", "sessions_attended_before_t3"]].fillna(0).astype(int))

# Attendance rate = attended / scheduled (left empty if no sessions were scheduled)
features["attendance_rate_before_t3"] = np.where(
    features["sessions_scheduled_before_t3"] > 0,
    features["sessions_attended_before_t3"] / features["sessions_scheduled_before_t3"],
    np.nan)

# Attendance groups for H2: 0-24%, 25-49%, 50-74%, 75-100%
features["attendance_group"] = pd.cut(features["attendance_rate_before_t3"] * 100,
                                      bins=[0, 25, 50, 75, 100.01], right=False,
                                      labels=["0-24%", "25-49%", "50-74%", "75-100%"])

print("Rows:", len(features))

# Check: only students referred before term 3 should have sessions before term 3
print("\nRows with sessions before term 3:", (features["sessions_scheduled_before_t3"] > 0).sum())
print("...of those, NOT referred before term 3:",
      ((features["sessions_scheduled_before_t3"] > 0) & ~features["referred_before_t3"]).sum())
print("Referred before term 3 but 0 sessions before term 3:",
      (features["referred_before_t3"] & (features["sessions_scheduled_before_t3"] == 0)).sum())

# Attendance groups among referred students
print("\nAttendance groups (referred before term 3):")
print(features.loc[features["referred_before_t3"], "attendance_group"].value_counts(dropna=False).sort_index())

Sessions before term 3: 239606 of 413370


Rows: 120000

Rows with sessions before term 3: 13538
...of those, NOT referred before term 3: 0
Referred before term 3 but 0 sessions before term 3: 0

Attendance groups (referred before term 3):
attendance_group
0-24%      1737
25-49%     3307
50-74%     4547
75-100%    3947
Name: count, dtype: int64


**Result:** Still 120,000 rows. 239,606 of the 413,370 sessions happened before
term 3 (term 1 + term 2, same as the cleaning notebook).

13,538 student-subject rows had at least one session before term 3. These are
exactly the students referred before term 3, and no one without a referral
had sessions. Every referred student had at least one session scheduled before
term 3, so no one is left out of H2 for having no sessions.

Attendance groups among referred students:
0-24% = 1,737, 25-49% = 3,307, 50-74% = 4,547, 75-100% = 3,947.

## Step 7 – Final checks and save
Before saving, we check that the table is complete: one row per student and
subject, no missing values , and no information
from term 3 except the target. Then we save it as features.csv for the
experiments and models.

In [31]:
# Put the columns in a clear order
cols = ["student_id", "subject_area", "school_code", "grade_level",
        "failed_t3",                                   # target (what we predict)
        "failed_t1", "failed_t2",                      # earlier results
        "prior_year_gpa", "gpa_missing", "gpa_group",  # past grades
        "referred_before_t3", "referred_during_t3", "referred_any", "referral_date", "referred_by",
        "sessions_scheduled_before_t3", "sessions_attended_before_t3",
        "attendance_rate_before_t3", "attendance_group",
        "bus_rider", "econ_disadvantaged", "english_learner", "iep"]
features = features[cols]

# Check 1: one row per student and subject
print("Rows:", len(features))
print("Duplicate student-subject rows:", features.duplicated(["student_id", "subject_area"]).sum())

# Check 2: missing values in each column
print("\nMissing values:")
print(features.isna().sum()[features.isna().sum() > 0])

# Check 3: no term-3 information other than the target
leaky = [c for c in features.columns if c in ["term_grade", "grade_pct", "absences"]]
print("\nTerm-3 grade columns in the table (should be none):", leaky)

# How many rows failed in term 3 (the target)
print("\nFailed in term 3:", features["failed_t3"].sum(),
      f"({features['failed_t3'].mean():.1%})")

# Save the table
features.to_csv("../data/features.csv", index=False)
print("\nSaved ../data/features.csv")

Rows: 120000
Duplicate student-subject rows: 0

Missing values:
prior_year_gpa                 3609
referral_date                105083
referred_by                  105083
attendance_rate_before_t3    106462
attendance_group             106462
dtype: int64

Term-3 grade columns in the table (should be none): []

Failed in term 3: 12237 (10.2%)



Saved ../data/features.csv


**Result:** The table has 120,000 rows with no duplicate student-subject rows.

Missing values are only where we expect them:
- prior GPA: 3,609 rows (new students)
- referral date and referred by: 105,083 rows (never referred)
- attendance: 106,462 rows (no sessions before term 3)

No term-3 grade columns are in the table except the target.

12,237 rows (10.2%) failed in term 3.

Saved as `data/features.csv`. This file is used by the experiments and models notebooks.